# 02 - 画像のベクトル化

01で作成した画像カタログをもとに、SigLIPでベクトル化を行う。

## 入力
- `data/images.duckdb` (image_catalog テーブル)

## 出力
- `data/images.duckdb` (image_embeddings テーブル + HNSWインデックス)

In [2]:
from pathlib import Path

import duckdb
from PIL import Image
from tqdm.notebook import tqdm

from image_vector_poc import SigLIPEmbedder

## 設定

In [3]:
# DuckDBファイルのパス
DB_PATH = Path("../data/images.duckdb")

# バッチサイズ
BATCH_SIZE = 32

# コミット間隔（何件ごとにコミットするか）
COMMIT_INTERVAL = 100

## モデルの初期化

In [4]:
# SigLIPモデルの読み込み
print("SigLIPモデルを読み込み中...")
embedder = SigLIPEmbedder(device="cuda")
print(f"モデル: {embedder.model_name}")
print(f"埋め込み次元: {embedder.embedding_dim}")

SigLIPモデルを読み込み中...


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

モデル: google/siglip-base-patch16-224
埋め込み次元: 768


## DuckDBテーブル作成

In [5]:
# データベース接続
conn = duckdb.connect(str(DB_PATH))

# VSS拡張を有効化
conn.execute("INSTALL vss; LOAD vss;")

# 埋め込みテーブル作成
embedding_dim = embedder.embedding_dim
conn.execute(f"""
    CREATE TABLE IF NOT EXISTS image_embeddings (
        id VARCHAR PRIMARY KEY,
        embedding FLOAT[{embedding_dim}],
        model_name VARCHAR,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")

print("テーブル作成完了")

テーブル作成完了


## 処理対象の確認

In [6]:
# カタログの総数
total_catalog = conn.execute("SELECT COUNT(*) FROM image_catalog").fetchone()[0]
print(f"カタログ総数: {total_catalog}")

# 既に処理済みの数
already_processed = conn.execute("SELECT COUNT(*) FROM image_embeddings").fetchone()[0]
print(f"処理済み: {already_processed}")

# 未処理の数
remaining = total_catalog - already_processed
print(f"未処理: {remaining}")

カタログ総数: 385
処理済み: 0
未処理: 385


In [7]:
# 未処理の画像を取得
pending_images = conn.execute("""
    SELECT c.id, c.file_path 
    FROM image_catalog c
    LEFT JOIN image_embeddings e ON c.id = e.id
    WHERE e.id IS NULL
    ORDER BY c.id
""").fetchall()

print(f"処理対象: {len(pending_images)} 件")

処理対象: 385 件


## ベクトル化処理

In [8]:
def process_batch(batch_items: list[tuple[str, str]], embedder, conn, model_name: str):
    """バッチ単位でベクトル化を行う"""
    ids = []
    images = []
    
    for id_, file_path in batch_items:
        try:
            img = Image.open(file_path).convert("RGB")
            ids.append(id_)
            images.append(img)
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
    
    if not images:
        return 0
    
    # バッチでベクトル化
    embeddings = embedder.embed_images(images)
    
    # DBに保存
    for id_, embedding in zip(ids, embeddings):
        conn.execute("""
            INSERT OR REPLACE INTO image_embeddings (id, embedding, model_name)
            VALUES (?, ?, ?)
        """, [id_, embedding.tolist(), model_name])
    
    return len(ids)

In [9]:
# バッチ処理でベクトル化
model_name = embedder.model_name
processed = 0
errors = 0

# プログレスバー
pbar = tqdm(total=len(pending_images), desc="ベクトル化中")

for i in range(0, len(pending_images), BATCH_SIZE):
    batch = pending_images[i:i + BATCH_SIZE]
    
    try:
        count = process_batch(batch, embedder, conn, model_name)
        processed += count
        errors += len(batch) - count
    except Exception as e:
        print(f"Batch error: {e}")
        errors += len(batch)
    
    pbar.update(len(batch))
    
    # 定期的にコミット
    if processed % COMMIT_INTERVAL == 0:
        conn.commit()

pbar.close()

# 最終コミット
conn.commit()

print(f"\n処理完了: 成功={processed}, エラー={errors}")

ベクトル化中:   0%|          | 0/385 [00:00<?, ?it/s]


処理完了: 成功=385, エラー=0


## HNSWインデックス作成

In [11]:
# 永続化されたDBでHNSWインデックスを使うための設定
conn.execute("SET hnsw_enable_experimental_persistence = true")

# 既存のインデックスを削除（あれば）
try:
    conn.execute("DROP INDEX IF EXISTS embedding_idx")
except:
    pass

# HNSWインデックス作成（コサイン類似度）
print("HNSWインデックスを作成中...")
conn.execute("""
    CREATE INDEX embedding_idx 
    ON image_embeddings 
    USING HNSW (embedding) 
    WITH (metric = 'cosine')
""")
print("HNSWインデックス作成完了")

HNSWインデックスを作成中...
HNSWインデックス作成完了


## 結果確認

In [12]:
# 埋め込み済み件数
embedding_count = conn.execute("SELECT COUNT(*) FROM image_embeddings").fetchone()[0]
print(f"埋め込み済み: {embedding_count} 件")

埋め込み済み: 385 件


In [13]:
# サンプル検索テスト
print("\nサンプル検索テスト:")
test_query = "a photo of nature"
query_embedding = embedder.embed_text(test_query)

results = conn.execute(f"""
    SELECT 
        e.id,
        c.file_name,
        c.category,
        array_cosine_distance(e.embedding, ?::FLOAT[{embedding_dim}]) as distance
    FROM image_embeddings e
    JOIN image_catalog c ON e.id = c.id
    ORDER BY distance
    LIMIT 5
""", [query_embedding.tolist()]).fetchall()

print(f"クエリ: '{test_query}'")
print("Top 5 結果:")
for id_, name, cat, dist in results:
    print(f"  {name} ({cat}) - distance: {dist:.4f}")


サンプル検索テスト:
クエリ: 'a photo of nature'
Top 5 結果:
  2026-01-03 13.24.30.jpg (KashiwaVillagePark2026) - distance: 0.9215
  2026-01-03 13.21.09.jpg (KashiwaVillagePark2026) - distance: 0.9238
  2026-01-03 13.31.40.jpg (KashiwaVillagePark2026) - distance: 0.9267
  2026-01-03 13.23.36.jpg (KashiwaVillagePark2026) - distance: 0.9274
  2026-01-03 13.47.33.jpg (KashiwaVillagePark2026) - distance: 0.9304


In [14]:
# 接続を閉じる
conn.close()
print(f"\n完了: {DB_PATH.resolve()}")


完了: /home/terapyon/dev/vibe-coding/image-vector-poc/data/images.duckdb
